In [ ]:
import pandas as pd
import numpy as np
import glob
import os
from google.colab import drive

# 1. Montar Google Drive para acceder a la carpeta 'bt'
drive.mount('/content/drive')

# 2. Configurar la ruta (Ajusta 'Mi Unidad/...' según donde esté tu carpeta bt)
path = '/content/drive/MyDrive/bt'
all_files = glob.glob(os.path.join(path, "*.csv"))

print(f"🔍 Buscando en: {path}")
print(f"✅ Se encontraron {len(all_files)} archivos. Iniciando fusión de Big Data...")

# 3. Fusión de Fragmentos (Cosiendo la Data)
li = []
for filename in all_files:
    try:
        # Detectamos el separador (algunos MT5 usan ',' otros ';')
        df = pd.read_csv(filename, index_col=None, header=0, sep=None, engine='python')
        li.append(df)
    except Exception as e:
        print(f"⚠️ Error en archivo {filename}: {e}")

# Crear el Mega-DataFrame
master_df = pd.concat(li, axis=0, ignore_index=True)

# 4. Limpieza y Ordenamiento Cronológico
# Convertimos la columna Time a formato fecha real
master_df['Time'] = pd.to_datetime(master_df['Time'])
master_df = master_df.sort_values(by='Time')

# Eliminamos duplicados por si el MT5 repitió datos en los agentes
master_df = master_df.drop_duplicates()

print(f"📊 Total de registros procesados: {len(master_df)}")

# 5. Filtrado de la "JUGADA MAESTRA" (Nivel 4/5)
# Basado en los parámetros de la Tesis ARC v42.2
igniciones = master_df[
    (master_df['Upsilon_Tension'] < 0.25) &
    (master_df['Confluence_Score'] >= 3) &
    (master_df['Tick_Speed'] > 35)
]

# 6. Cálculo de Eficiencia (¿Cuánto ganamos?)
print("\n--- 🔱 RESULTADOS DE LA TESIS ARC ---")
print(f"🎯 Eventos de Alta Probabilidad (Nivel 4+): {len(igniciones)}")

if len(igniciones) > 0:
    avg_energy = igniciones['Energy_K'].mean()
    print(f"⚡ Energía Promedio en Ignición: {avg_energy:.4f}")
    print(f"📈 Sugerencia: Operar con Tick_Speed > {master_df['Tick_Speed'].mean()*1.4:.1f}")

# 7. Guardar el archivo para siempre
master_df.to_csv('/content/drive/MyDrive/MASTER_BT_ARC_V42.csv', index=False)
print("\n💾 Archivo UNIFICADO guardado en tu Drive como: MASTER_BT_ARC_V42.csv")

Mounted at /content/drive
🔍 Buscando en: /content/drive/MyDrive/bt
✅ Se encontraron 29 archivos. Iniciando fusión de Big Data...
